# OpenStreetMap

In [ ]:
import geopandas as gpd

## Carga de datos

In [3]:
path_pbf = 'osm/andalucia-260305.osm.pbf'

# 1. Extraer el polígono de la Provincia de Cádiz (Admin Level 6)
provincia_cadiz = gpd.read_file(
    path_pbf,
    engine='pyogrio',
    layer='multipolygons',
    where="admin_level = '6' AND name = 'Cádiz'"
)

# 2. Extraer todos los municipios (Admin Level 8) de Andalucía
municipios_and = gpd.read_file(
    path_pbf,
    engine='pyogrio',
    layer='multipolygons',
    where="admin_level = '8'"
)

In [ ]:
provincia_cadiz

,osm_id,osm_way_id,name,type,aeroway,amenity,admin_level,barrier,boundary,building,...,man_made,military,natural,office,place,shop,sport,tourism,other_tags,geometry
0,349017,None,Cádiz,boundary,None,None,6,None,administrative,None,...,None,None,None,None,province,None,None,None,"""ISO3166-2""=>""ES-CA"",""ine:provincia""=>""11"",""na...","MULTIPOLYGON (((-5.27354 36.292, -5.2736 36.29..."


In [ ]:
# Es vital que ambos tengan el mismo Sistema de Referencia (CRS)
# Generalmente OSM viene en EPSG:4326
municipios_and = municipios_and.to_crs(provincia_cadiz.crs)

# Realizamos la unión espacial
# 'inner' para quedarnos solo con los que coinciden
# 'within' asegura que el municipio esté dentro de la provincia
municipios_cadiz = gpd.sjoin(municipios_and, provincia_cadiz, predicate='within', how='inner')

# Limpiar columnas duplicadas tras el join (opcional)
municipios_cadiz = municipios_cadiz[['name_left', 'geometry', 'admin_level_left']]
municipios_cadiz.columns = ['name', 'geometry', 'admin_level']

In [ ]:
lista_limites = []

for idx, row in municipios_cadiz.iterrows():
    # .bounds devuelve (minx, miny, maxx, maxy)
    minx, miny, maxx, maxy = row['geometry'].bounds
    
    lista_limites.append({
        'municipio': row['name'],
        'bbox': [minx, miny, maxx, maxy]
    })

# Ejemplo de visualización rápida
print(f"Total municipios encontrados en Cádiz: {len(municipios_cadiz)}")
print(f"Límites de {lista_limites[0]['municipio']}: {lista_limites[0]['bbox']}")

Total municipios encontrados en Cádiz: 45
Límites de Paterna de Rivera: [-5.8954597, 36.4971526, -5.8285846, 36.5382019]
